In [0]:
from pyspark.sql.functions import current_timestamp
import re

In [0]:
# Read the CSV file with header and inferSchema options enabled
df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/Volumes/bosch/bronze/dataset3/vehicles.csv")
)

In [0]:
def clean_col(c: str) -> str:
    # Remove leading/trailing spaces and convert to lowercase
    c = c.strip().lower()
    # Replace forbidden characters with underscore
    c = re.sub(r"[ ,;{}()\n\t=]+", "_", c)
    # Remove any remaining non-alphanumeric or underscore characters
    c = re.sub(r"[^a-z0-9_]", "", c)
    # Normalize consecutive underscores and remove leading/trailing underscores
    c = re.sub(r"_+", "_", c).strip("_")
    return c

# Rename DataFrame columns using the clean_col function
df = df.toDF(*[clean_col(c) for c in df.columns])

# Add ingestion timestamp column
df = df.withColumn("ingestion_ts", current_timestamp())

In [0]:
df.write.format("delta") \
    .mode("append") \
    .saveAsTable("bronze.epa_vehicles_raw")